# Chunking Strategies & Retrieval

> **How should I turn a document into retrievable pieces without throwing away the context that makes those pieces meaningful?**

There is no universally "best" chunking strategy. The right choice depends on the document, the questions users ask, the embedding model, the context window, and how much latency/cost you can afford.

We will start with simple baselines, move toward structure-aware and semantic methods, then look at newer context-preserving approaches such as **parent-child retrieval, contextual retrieval, late chunking, and hierarchical retrieval**.

The important distinction throughout the notebook is:

- **Chunking** decides *what pieces exist*.
- **Indexing/enrichment** decides *what representation of those pieces is searchable*.
- **Retrieval** decides *which pieces to fetch*.
- **Reranking** decides *which retrieved pieces are actually most relevant*.

That distinction matters because several techniques commonly called "chunking strategies" are actually retrieval architectures.

## 0. Setup

Expected files in `src/`:

- `company_policy.txt`
- `annual_report.md`
- `technical_manual.html`
- `python_project.py`
- `research_paper.pdf`
- `product_catalog.json`

In [3]:
import os
from pathlib import Path

# Keep Hugging Face downloads disabled if the required model is already cached locally.
os.environ["HF_HUB_OFFLINE"] = "1"

from dotenv import load_dotenv
load_dotenv()


DATA_DIR = Path("src")
print("Data directory:", DATA_DIR.resolve())

Data directory: C:\Users\TEK-UP\Desktop\TEK-UP\AI_Engineering_Course\labs\04-rag\src


## 1. Fixed-size character chunking

This is the simplest possible baseline.

We say: "give me 300 characters, then the next 300, and so on."

It is useful because it gives us something simple to compare against. It is usually not what we want in a production RAG system because the boundary has no relationship to meaning or document structure.

**Good for:** a baseline and very controlled experiments.

**Weakness:** it can cut a sentence, list item, code block, or table in the middle.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text = (DATA_DIR / "company_policy.txt").read_text(encoding="utf-8")

char_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=300,
    chunk_overlap=0,
)
char_chunks = char_splitter.split_text(text)

print(f"{len(char_chunks)} chunks")
for c in char_chunks[:7]:
    
    print("-", c[:120].replace("\n", " "), "...")

11 chunks
- Remote Work and Code of Conduct Policy  This policy applies to all employees, contractors, and interns working at or on  ...
- aising concerns.  Remote work is available to employees whose role does not require regular physical presence at a compa ...
- etting. Core collaboration hours are 10:00 to 16:00 local time, during which employees should be reachable via the compa ...


## 2. Recursive character chunking

This is a much more practical generic baseline.

Instead of cutting immediately, it tries increasingly smaller separators. With the default configuration, it prefers paragraph-like boundaries, then smaller boundaries, and finally a hard character-level split if necessary.

The key idea is:

> **Keep meaningful text together when possible, but still enforce a maximum size.**

This is why recursive splitting is a strong default for ordinary prose.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
)

recursive_chunks = recursive_splitter.split_text(text)

print(f"{len(recursive_chunks)} chunks")
for c in recursive_chunks[:7]:
    print("-", c[:120].replace("\n", " "), "...")

15 chunks
- Remote Work and Code of Conduct Policy ...
- This policy applies to all employees, contractors, and interns working at or on behalf of the company, regardless of loc ...
- Remote work is available to employees whose role does not require regular physical presence at a company site, subject t ...
- in an office setting. Core collaboration hours are 10:00 to 16:00 local time, during which employees should be reachable ...
- Employees working from a personal device or home network are responsible for keeping that equipment reasonably secure. T ...
- Wi-Fi networks for any task involving customer data or internal systems. Any suspected security incident, including a lo ...
- All employees are expected to treat colleagues, clients, and partners with respect, regardless of role, seniority, or ba ...


## 3. Token-based chunking

Characters are only a proxy for what an LLM or embedding model sees.

Token-based splitting lets us express the budget in tokens. This is useful when the model's context limit, embedding limit, or cost is expressed in tokens.

A useful rule for students:

> **Character size is a convenient engineering approximation; token size is closer to the model's actual constraint.**

Do not assume that one token equals one word or one character.

In [7]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=100,
    chunk_overlap=10,
)

token_chunks = token_splitter.split_text(text)

print(f"{len(token_chunks)} chunks")
for c in token_chunks[:7]:
    print("-", c[:120].replace("\n", " "), "...")

7 chunks
- Remote Work and Code of Conduct Policy  This policy applies to all employees, contractors, and interns working at or on  ...
-  the same standard of availability, responsiveness, and output as they would in an office setting. Core collaboration ho ...
- . This includes using the company-approved password manager, enabling full-disk encryption where supported, and avoiding ...
-  with respect, regardless of role, seniority, or background. Harassment, discrimination, and retaliation of any kind are ...
-  the legal team. This obligation continues after an employee's departure from the company. Employees who are unsure whet ...
-  action, up to and including termination.  Employees who believe this policy has been violated, or who have concerns abo ...
-   This policy is reviewed annually and may be updated to reflect changes in applicable law, company structure, or workin ...


## 4. Sentence-based chunking

Sentence boundaries are a useful middle ground.

Instead of asking an embedding model where the meaning changes, we first keep complete sentences and then group them until a size budget is reached.

This is cheaper and more predictable than semantic chunking.

For a classroom demo, a simple regex is enough. For production, a proper NLP sentence segmenter can be preferable for difficult punctuation and multilingual text.

In [8]:
import re

def split_sentences(text):
    # Simple teaching implementation; not a full linguistic sentence parser.
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

def sentence_chunks(text, max_chars=300):
    sentences = split_sentences(text)
    chunks = []
    current = []

    for sentence in sentences:
        candidate = " ".join(current + [sentence])
        if current and len(candidate) > max_chars:
            chunks.append(" ".join(current))
            current = [sentence]
        else:
            current.append(sentence)

    if current:
        chunks.append(" ".join(current))

    return chunks

sentence_based_chunks = sentence_chunks(text, max_chars=300)

print(f"{len(sentence_based_chunks)} chunks")
for c in sentence_based_chunks[:3]:
    print("-", c[:120], "...")

15 chunks
- Remote Work and Code of Conduct Policy

This policy applies to all employees, contractors, and interns working at or on  ...
- It covers expectations around remote work arrangements, professional conduct, data security, and the process for raising ...
- Employees working remotely are expected to maintain the same standard of availability, responsiveness, and output as the ...


## 5. Overlap and sliding windows

Overlap is often described as a "chunking strategy", but it is better understood as a **parameter applied to another strategy**.

For example:

- recursive chunks + 20% overlap
- token chunks + 10% overlap
- sentence windows + one-sentence overlap

Why use it?

Because an answer may sit exactly across a chunk boundary.

Why not make overlap huge?

Because duplicated text increases:

- index size
- embedding cost
- retrieval redundancy
- the number of near-duplicate chunks passed to the LLM

So overlap is a trade-off, not automatically a good thing.

In [ ]:
# Visualize the idea with plain text windows.
def sliding_windows(items, window_size=3, step=2):
    return [
        items[i:i + window_size]
        for i in range(0, max(1, len(items) - window_size + 1), step)
    ]

sentences = split_sentences(text)
windows = sliding_windows(sentences, window_size=3, step=2)

for i, window in enumerate(windows[:3], 1):
    print(f"WINDOW {i}")
    print(" ".join(window))
    print()

## 6. Markdown structure-aware chunking

Markdown gives us information for free: headings already describe the document's hierarchy.

Instead of pretending that the document is an undifferentiated string, we can split around headings and carry those headings as metadata.

This is often better than semantic chunking for well-structured documentation.

**Important:** a section can still be too large. Structure-aware chunking and recursive splitting are complementary:

1. split by document structure;
2. preserve the section metadata;
3. recursively split oversized sections.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_text = (DATA_DIR / "annual_report.md").read_text(encoding="utf-8")

md_splitter = MarkdownHeaderTextSplitter(
    [("#", "h1"), ("##", "h2"), ("###", "h3")]
)

md_sections = md_splitter.split_text(md_text)

print(f"{len(md_sections)} structured sections")
for doc in md_sections[:3]:
    print("-", doc.metadata)
    print(doc.page_content[:150].replace("\n", " "), "...")
    print()

## 7. HTML structure-aware chunking

HTML has an even richer structure than plain text.

LangChain provides header-aware and semantic-preserving HTML splitters. A semantic-preserving splitter can keep HTML elements together and then fall back to recursive splitting when a section becomes too large.

This is a good example of an important general pattern:

> **Use the document's native structure first; use generic splitting as a fallback.**

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

headers_to_split_on = [
    ("h1", "h1"),
    ("h2", "h2"),
    ("h3", "h3"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on)
html_chunks = html_splitter.split_text_from_file(
    str(DATA_DIR / "technical_manual.html")
)

print(f"{len(html_chunks)} HTML sections")
for doc in html_chunks[:3]:
    print("-", doc.metadata)
    print(doc.page_content[:150].replace("\n", " "), "...")
    print()

## 8. Code-aware chunking

Source code is not ordinary prose.

A useful boundary might be:

- class
- method/function
- control-flow block
- import section
- module

LangChain has language-aware recursive splitting for many programming languages. It uses language-specific separators rather than treating the file as arbitrary English text.

The important teaching point is not "use this one splitter". It is:

> **The best boundary depends on the structure of the data.**

In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

code_text = (DATA_DIR / "python_project.py").read_text(encoding="utf-8")

code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=300,
    chunk_overlap=30,
)

code_chunks = code_splitter.split_text(code_text)

print(f"{len(code_chunks)} code chunks")
for c in code_chunks[:7]:
    print("---")
    print(c[:300])

## 9. JSON and hierarchical data

JSON should not be treated as a long string if the hierarchy carries meaning.

For example:

```text
catalog
  └── product
       ├── name
       ├── description
       └── specifications
```

A structure-aware splitter can preserve that hierarchy while producing chunks that remain valid representations of the underlying data.

This same principle applies to XML, YAML and nested API responses.

In [ ]:
import json as json_lib
from langchain_text_splitters import RecursiveJsonSplitter

with open(DATA_DIR / "product_catalog.json", encoding="utf-8") as f:
    catalog = json_lib.load(f)

json_splitter = RecursiveJsonSplitter(max_chunk_size=300)
json_chunks = json_splitter.split_text(catalog)

print(f"{len(json_chunks)} JSON chunks")
for c in json_chunks[:3]:
    print("-", c[:250])

## 10. Semantic chunking

Semantic chunking asks a different question:

> "Where does the meaning change enough that starting a new chunk makes sense?"

A common implementation embeds neighboring sentences and creates a boundary when their semantic similarity changes significantly.

This can work well for prose whose topic changes gradually, but it is more expensive than rule-based splitting and introduces model-dependent behavior.

It is **not automatically better** than recursive or structure-aware splitting. Evaluate it on your own retrieval task.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

semantic_splitter = SemanticChunker(embeddings)
semantic_chunks = semantic_splitter.split_text(md_text)

print(f"{len(semantic_chunks)} semantic chunks")
for c in semantic_chunks[:3]:
    print("-", c[:120].replace("\n", " "), "...")

## 11. Parent-child / small-to-big retrieval

This is where the terminology gets important.

**Parent-child retrieval is not simply another way of cutting text.** It is a retrieval architecture.

The idea is:

1. split a document into relatively large parent sections;
2. split each parent into smaller child chunks;
3. embed/search the small children;
4. when a child matches, return its parent (or a suitable larger context).

This separates two competing goals:

- small chunks are easier to match precisely;
- larger chunks preserve context for generation.

Modern LangChain exposes `ParentDocumentRetriever` in the `langchain-classic` package.

In [ ]:
# Current LangChain packaging: advanced retrievers are in langchain-classic.
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

PARENT_COLLECTION = "parent_child_demo"

client = QdrantClient(url="http://localhost:6333")

if client.collection_exists(PARENT_COLLECTION):
    client.delete_collection(PARENT_COLLECTION)

client.create_collection(
    PARENT_COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=0,
)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=PARENT_COLLECTION,
    embedding=embeddings,
)

docstore = InMemoryStore()

pc_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

pc_retriever.add_documents([Document(page_content=md_text)])

results = pc_retriever.invoke("what does the report say about deep learning?")

print(f"Returned {len(results)} parent chunk(s)")
if results:
    print(results[0].page_content[:500])

## 12. Contextual Retrieval

A small chunk can be technically correct but still be difficult to retrieve because it has lost the context around it.

For example, a chunk may say:

> "The company increased revenue by 3%."

The chunk itself does not tell us:

- which company?
- which quarter?
- which business unit?
- compared with what?

**Contextual Retrieval** addresses this by generating a short, chunk-specific context from the document and prepending it to the chunk before indexing.

Conceptually:

```text
document context + chunk-specific context + original chunk
                         ↓
                 embedding / BM25 index
```

Anthropic reported improvements from combining contextual embeddings and contextual BM25, with further gains from reranking. Those numbers are benchmark-specific; they are evidence that the technique can help, not a guarantee for every dataset.

This is best understood as **index-time context enrichment**, not as a new text splitter.

### Contextual Retrieval — teaching pseudocode

```python
for chunk in chunks:
    context = llm(
        f"""Given the whole document and this chunk,
        write a short description that explains where this chunk
        belongs and what entities or concepts it refers to."""
    )

    indexed_text = context + "\n" + chunk
    embed(indexed_text)
    bm25.add(indexed_text)
```

At query time, retrieve using the enriched representation, but keep the original chunk available for the final answer.

**Practical warning:** generating context for every chunk adds LLM cost and latency during ingestion. It should be evaluated against a cheaper baseline first.

## 13. Late chunking

Late chunking attacks the same lost-context problem from a different direction.

Traditional pipeline:

```text
document
   ↓
chunk
   ↓
embed each chunk independently
```

Late chunking:

```text
document
   ↓
long-context embedding model
   ↓
contextualized token embeddings
   ↓
apply chunk boundaries
   ↓
pool token embeddings into chunk embeddings
```

The chunk boundaries still matter. The difference is **when** the embedding model sees the context.

This requires an embedding model capable of processing a sufficiently long input and exposing token-level representations. It is therefore not a normal `TextSplitter` feature.

For very long documents, the idea can be extended with overlapping macro-chunks that fit the embedding model's context window.

### Late chunking: what to remember

Late chunking is especially interesting when:

- the document is internally coherent;
- small chunks lose references such as "it", "this method", or "the company";
- the embedding model supports long contexts.

It is less attractive when a single document contains many unrelated topics, because global context can introduce noise.

So again:

> **A newer technique is not automatically a better technique. The document and retrieval task decide.**

## 14. Hierarchical retrieval and RAPTOR-style indexing

Another direction is to represent the same document at multiple levels:

```text
                 document summary
                       │
              ┌────────┴────────┐
          section summary    section summary
             │      │            │
          chunk   chunk        chunk
```

RAPTOR recursively embeds, clusters and summarizes chunks to construct a tree with multiple levels of abstraction. Retrieval can then happen over both detailed chunks and higher-level summaries.

This is useful for questions that require a global understanding of a long document rather than one isolated sentence.

Again, this is an **index/retrieval architecture**, not simply a text splitter.

For a first RAG course, it is better to teach the idea and architecture before asking students to implement the complete recursive indexing pipeline.

## 15. Tables, PDFs and layout-heavy documents

This is one of the most important practical gaps in simple chunking demos.

A PDF is not necessarily "a text file with pages".

Real PDFs can contain:

- multi-column text
- tables
- headers and footers
- footnotes
- figures
- captions
- scanned pages
- text positioned in arbitrary coordinates

If extraction destroys that structure, a perfect chunking algorithm cannot reconstruct it reliably.

For layout-heavy documents, the pipeline should be thought of as:

```text
PDF / HTML / DOCX
       ↓
structure-aware extraction
       ↓
elements: paragraphs / headings / tables / lists / captions
       ↓
element-aware chunking
       ↓
metadata
       ↓
embedding + retrieval
```

Tables deserve particular attention: splitting a table row away from its header can make the values almost meaningless.

For advanced PDF work, use a document parser that exposes layout/elements first, then design chunking around those elements.

## 16. Metadata is part of chunking

A chunk should not only be:

```python
{"text": "..."}
```

A useful production representation is closer to:

```python
{
    "text": "...",
    "source": "annual_report.md",
    "document_id": "annual_report_2025",
    "section": "Financial Results",
    "page": 12,
    "position": 18420,
    "parent_id": "section_07"
}
```

Metadata can later support:

- filtering by document
- filtering by section
- access control
- citations
- parent retrieval
- deduplication
- debugging bad retrieval
- displaying the source to the user

This is why **chunking and metadata extraction should usually be designed together**.

# Retrieval

Chunking only creates the searchable units. We now compare different ways of finding those units.

The three basic retrieval families in this notebook are:

1. **dense semantic retrieval** — meaning similarity;
2. **BM25** — lexical matching;
3. **hybrid retrieval** — combine both.

A modern production pipeline often adds **reranking** after initial retrieval.

## 17. Connect to the existing Qdrant collection

This assumes the collection from the earlier RAG lab already exists.

In [ ]:
from langchain_qdrant import QdrantVectorStore

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "my_first_rag"

vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    url=QDRANT_URL,
    collection_name=COLLECTION_NAME,
)

## 18. Semantic vector search

Dense retrieval embeds the query and searches for chunks with nearby vectors.

It is good when the query and document use different words but express similar ideas.

Example:

```text
query: "How do I remember a conversation?"
document: "The application stores previous messages between turns."
```

The wording is different, but the meaning is related.

In [ ]:
def semantic_search(query, k=5):
    return vector_store.similarity_search(query, k=k)

query = "what does the deep learning module cover?"

semantic_results = semantic_search(query)

for r in semantic_results:
    print("-", r.page_content[:120].replace("\n", " "), "...")

## 19. BM25 keyword search

BM25 is lexical retrieval. It rewards useful term matches and is particularly valuable when exact strings matter:

- product IDs
- error codes
- function names
- acronyms
- legal terms
- technical identifiers

Dense embeddings and BM25 fail in different ways, which is one reason hybrid retrieval is so useful.

In [ ]:
from rank_bm25 import BM25Okapi

# Pull the indexed text from Qdrant for this classroom implementation.
all_points = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10000,
)[0]

bm25_corpus_docs = [
    p.payload["page_content"]
    for p in all_points
]

bm25_corpus = [doc.lower().split() for doc in bm25_corpus_docs]
bm25 = BM25Okapi(bm25_corpus)

def bm25_search(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    ranked_idx = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )[:k]
    return [bm25_corpus_docs[i] for i in ranked_idx]

bm25_results = bm25_search(query)

for r in bm25_results:
    print("-", r[:120].replace("\n", " "), "...")

## 20. Hybrid retrieval with Reciprocal Rank Fusion

Dense and BM25 scores are not naturally comparable:

- cosine similarity has one scale;
- BM25 has another.

Instead of trying to add the raw scores directly, we can combine their **rank positions**.

Reciprocal Rank Fusion (RRF) gives a result a contribution based on where it appears in each ranking.

This is simple, robust, and easy to explain.

In [ ]:
def reciprocal_rank_fusion(rankings, rrf_k=60):
    scores = {}

    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (rrf_k + rank)

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

semantic_ranking = [
    r.page_content
    for r in semantic_search(query, k=10)
]

bm25_ranking = bm25_search(query, k=10)

fused = reciprocal_rank_fusion(
    [semantic_ranking, bm25_ranking]
)

print("Hybrid top 5:")
for document, score in fused[:5]:
    print(f"- {score:.4f}", document[:100].replace("\n", " "), "...")

## 21. Reranking: another missing piece

Initial retrieval is usually optimized for **recall**: retrieve a reasonably broad candidate set.

A reranker then asks a more expensive question:

> "Given this query and these candidate passages, which passages are actually the most relevant?"

Typical pipeline:

```text
query
  ↓
BM25 + dense retrieval
  ↓
top 20–100 candidates
  ↓
reranker
  ↓
top 5–20 passages
  ↓
LLM
```

This is not chunking either. It is a second-stage retrieval operation.

It is especially useful when the first-stage retriever produces several plausible but only loosely relevant chunks.

# 22. A practical decision tree

Do not begin by asking "Which chunk size is state of the art?"

Start with the document.

### Plain prose

Start with:

```text
RecursiveCharacterTextSplitter
```

Then evaluate token size and overlap.

### Well-structured Markdown / HTML

Prefer:

```text
structure-aware splitting
        +
recursive fallback for oversized sections
```

### Source code

Use:

```text
language-aware splitting
```

### JSON / hierarchical data

Use:

```text
structure-aware JSON splitting
```

### Prose with strong topic shifts

Experiment with:

```text
semantic chunking
```

### Small matching units but larger answer context

Try:

```text
parent-child / small-to-big retrieval
```

### Chunks lose document context

Experiment with:

```text
contextual retrieval
```

or, when the embedding model supports it:

```text
late chunking
```

### Very long documents with questions at different levels

Consider:

```text
hierarchical retrieval / RAPTOR-style indexing
```

### Tables / complex PDFs

Fix **document extraction and layout preservation first**. Do not expect a text splitter to repair destroyed structure.

# 23. What is actually "state of the art"?

There is no single state-of-the-art chunker.

Current RAG practice is moving from:

```text
"pick a chunk size"
```

toward:

```text
document structure
      +
appropriate segmentation
      +
context preservation
      +
metadata
      +
hybrid retrieval
      +
reranking
      +
evaluation
```

Recent work on contextual retrieval and late chunking is particularly relevant because both address the same fundamental weakness of naive chunk embeddings: **a chunk can be too small to carry the context needed to understand or retrieve it**.

At the same time, evaluations show that these newer approaches can have trade-offs. For example, late chunking benefits from coherent documents but can introduce irrelevant global context when a document contains unrelated material.

So the modern lesson is not:

> "Use semantic/late/contextual chunking everywhere."

It is:

> **Choose the simplest representation that preserves the information your retrieval task needs, then measure it.**

# 24. Evaluation: stop guessing

A chunking strategy should be evaluated with the retrieval task it serves.

A simple experiment:

1. Prepare a fixed set of questions.
2. Annotate which chunks/documents contain the answer.
3. Run several chunking configurations.
4. Keep the embedding model and retrieval settings controlled.
5. Compare retrieval metrics.

Useful metrics include:

- **Recall@k** — did a relevant chunk appear in the top k?
- **Precision@k** — how much of the top k was relevant?
- **MRR** — how early was the first relevant result?
- **nDCG@k** — how well were graded relevance judgments ordered?

Then evaluate the final RAG answer separately. A retriever can have good recall while the final answer is still poor.

A useful classroom experiment is:

```text
recursive 300 / overlap 30
recursive 600 / overlap 60
token 200 / overlap 20
sentence 5-sentence windows
semantic
parent-child
```

Keep the rest of the pipeline constant and compare Recall@5 / Recall@10.

That turns "chunking feels better" into an actual engineering experiment.

# 25. Final mental model

When building RAG, think in this order:

```text
1. What kind of document is this?
             ↓
2. What structure can I preserve?
             ↓
3. What should one searchable unit represent?
             ↓
4. How much context does that unit need?
             ↓
5. How should I represent it for retrieval?
             ↓
6. How should I retrieve candidates?
             ↓
7. Should I rerank them?
             ↓
8. How will I evaluate the choice?
```

The most important practical takeaway:

> **Chunking is not about making pieces that are merely small enough. It is about creating retrieval units that are small enough to find precisely, but rich enough to mean something.**

# Sources and further reading

These references were checked while updating the notebook:

- LangChain text splitters reference:
  https://reference.langchain.com/python/langchain-text-splitters
- LangChain Markdown splitters:
  https://reference.langchain.com/python/langchain-text-splitters/markdown
- LangChain HTML splitters:
  https://reference.langchain.com/python/langchain-text-splitters/html
- LangChain ParentDocumentRetriever:
  https://reference.langchain.com/python/langchain-classic/retrievers/parent_document_retriever
- Anthropic — Contextual Retrieval:
  https://www.anthropic.com/engineering/contextual-retrieval
- Jina AI — Late Chunking:
  https://jina.ai/news/late-chunking-in-long-context-embedding-models/
- Jina AI — Long-context embeddings and chunking comparisons:
  https://jina.ai/en-US/news/still-need-chunking-when-long-context-models-can-do-it-all/
- RAPTOR, ICLR 2024:
  https://proceedings.iclr.cc/paper_files/paper/2024/hash/8a2acd174940dbca361a6398a4f9df91-Abstract-Conference.html

Research note:
The benchmark results reported by the linked papers/blogs are specific to their datasets, models and experimental settings. They should motivate experiments, not be treated as universal performance guarantees.